In [2]:
%pip install -U ddgs

  Using cached primp-0.15.0-cp38-abi3-win_amd64.whl.metadata (13 kB)
  Using cached fake_useragent-2.2.0-py3-none-any.whl.metadata (17 kB)
  Using cached brotli-1.2.0-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
  Using cached socksio-1.0.0-py3-none-any.whl.metadata (6.1 kB)
Using cached fake_useragent-2.2.0-py3-none-any.whl (161 kB)
Using cached socksio-1.0.0-py3-none-any.whl (12 kB)
Using cached primp-0.15.0-cp38-abi3-win_amd64.whl (3.1 MB)
Using cached brotli-1.2.0-cp311-cp311-win_amd64.whl (369 kB)

   ------------------------ --------------- 3/5 [fake-useragent]
   ---------------------------------------- 5/5 [ddgs]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import time
from typing import TypedDict, Optional
from dotenv import load_dotenv, find_dotenv
import mysql.connector
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_upstage import ChatUpstage
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, END
from langchain_teddynote.graphs import visualize_graph
# 환경 변수 로드
load_dotenv(find_dotenv())

# DB 설정
DB_CONFIG = {
    'host': os.getenv('DB_HOST'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME')
}

# LLM 설정 (Upstage Solar 우선, 없으면 OpenAI)

#llm = ChatOpenAI(model="gpt-5-mini", base_url="https://gms.ssafy.io/gmsapi/api.openai.com/v1/chat/completions", api_key=os.getenv("OPENAI_API_KEY"))
if os.getenv("UPSTAGE_API_KEY"):
    llm = ChatUpstage(model="solar-pro2")
# else:

# 검색 도구 설정
search_tool = DuckDuckGoSearchRun()


In [18]:

# 상태 정의
class EtlState(TypedDict):
    content_id: int
    title: str
    address: str
    search_results: Optional[str]
    generated_overview: Optional[str]

# 노드 함수 정의
def search_node(state: EtlState):
    """관광지 정보를 DuckDuckGo로 검색합니다."""
    # 검색 품질을 위해 검색어 구체화
    query = f"대한민국 {state['address']} {state['title']} 관광지 역사 특징 소개"
    print(f"Searching for: {query}")
    try:
        # DuckDuckGo 실행
        result = search_tool.invoke(query)
        context = result
    except Exception as e:
        print(f"Search failed: {e}")
        context = ""
    
    return {"search_results": context}

def summarize_node(state: EtlState):
    """검색 결과를 바탕으로 소개글을 작성합니다."""
    context = state.get("search_results", "")
    if not context or "No results" in context:
        return {"generated_overview": "정보를 찾을 수 없습니다."}

    # 프롬프트 강화: DuckDuckGo 결과가 Tavily보다 노이즈가 있을 수 있으므로 필터링 지침 강화
    prompt = f"""
    당신은 한국의 관광지 데이터를 정제하고 소개하는 전문 여행 에디터입니다.
    아래의 검색 결과 텍스트에는 해당 관광지와 무관하거나 광고성 내용이 섞여 있을 수 있습니다.
    오직 '{state['title']}'(주소: {state['address']})와 관련된 정확한 사실만을 걸러내어 매력적인 소개글을 작성해 주세요.
    
    [검색 결과]
    {context}
    
    [작성 가이드]
    1. 분량: 500자 ~ 700자 사이로 작성하세요.
    2. 어조: 친절하고 전문적인 '해요체'를 사용하세요. 가독성을 높여주세요.
    3. 내용 구성:
       - 도입: 해당 관광지가 어떤 곳인지 한 문장으로 정의
       - 본문: 주요 볼거리, 역사적 배경, 체험 요소 등 구체적 특징 서술
       - 마무리: 방문객에게 주는 팁이나 추천 멘트
    4. 제약 사항:
       - **단어** 같은 강조 표현은 사용하지마세요.
       - 검색 결과에 정보가 부족하여 내용을 구성할 수 없다면, 솔직하게 "정보 부족으로 작성 불가"라고 판단하고 빈 문자열을 반환하지 말고, 가지고 있는 일반적인 지식이 있다면 이를 활용하되 사실 여부가 불확실하면 생략하세요.
       - 검색 결과라는 말을 절대로 포함하지 마세요. 사용자가 해당 글을 읽고 검색을 통해 나온 결과임을 알 수 있으면 안됩니다.
       - 웹사이트 메뉴명이나 관련 없는 뉴스 헤드라인 등은 철저히 배제하세요.
       - 지역명(서울특별시, 경기도, 대구 등)에 대한 정보는 절대로 포함해서는 안됩니다. 
       - 정보가 부족하다는 답변을 하면 안됩니다. 어떻게든 내용을 채우세요. 그럴듯하게 적어도 됩니다.

    [예시1]
    국립민속박물관은 우리나라 전통 생활양식과 관련된 4,000여 점의 민속자료가 전시되어 있는 국내 유일의 민속생활사 관련 국립박물관이다. 본관과 파주관이 있는데 본관은 경복궁 경내에 있다. 본관 안에는 어린이박물관이 있다.국립민속박물관은 민속문화의 연구·수집·보존과 문화교육 및 생활문화 전시로 전통문화에 대한 올바른 인식을 통한 민족적 자긍심을 일깨울 수 있는 문화교육의 현장이다. 전통과 현대가 어우른 문화의 전당으로 3개의 상설전시실과 야외 전시, 2개의 기획전시실이 있다. 상설 전시 [1전시실]에서는 한국인의 하루, [2전시실]에서는 한국인의 일 년, [3전시실]에서는 한국인의 일생에 대해서 전시한다. 야외 전시에는 열두 띠 동상과 연자방아, 장승 동상 등이 전시되어 있고, 1970~1980년대 서울 지역 동네 골목을 재현한 [추억의 거리]가 재현되어 있다.다양한 문화교육 프로그램과 행사는 홈페이지의 [교육안내∙신청]을 통해 선착순으로 참여할 수 있다. 설날이나 추석과 같은 전통 명절에도 전통 놀이와 공연 등이 다채롭게 진행된다. 주한 외국인을 위한 체험행사도 진행하여 한국문화알기에 도움을 준다.국립민속박물관 어린이박물관은 우리나라 전통생활양식에 관련하여 어린이들에게 알기 쉽게 설명해 주고 체험해 볼 수 있도록 전시해 놓은 박물관이다. 어린이박물관은 홈페이지를 통해 1인당 1일 1회만 전시예약이 가능하고 사전예약을 해야 관람할 수 있다.

    [예시2]
    궁동저수지생태공원은 옛 궁동저수지의 열악한 환경 개선과 더불어 친환경적인 공간을 조성하여 구로구 지역 주민에게 휴식 공간을 제공하고자 조성되었다. 이곳은 생태탐방로, 산책로, 전통 한식 정자, 휴게시설, 체육시설 등이 있어 산책하기 좋으며 25,000여 본의 다양한 꽃과 나무가 식재되어 있다. 동식물을 관찰할 수 있는 공간과 어린이놀이터도 조성되어 있다. 생태연못에는 100여 마리의 비단잉어들이 저수지에 노닐고 있는 등 서울에서 보기 드문 아름다운 풍경과 생태 환경을 자랑한다. 궁동저수지생태공원에는 봄에는 철쭉이 피고 여름에는 연꽃이, 가을에는 갈대가, 겨울에는 얼어버린 저수지의 풍경이 사계절 내내 아름다운 곳이다. 이곳과 인접하여 정선옹주묘역이 있으므로 연계해서 둘러보는 것도 좋다.
    """
    
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        content = response.content.strip()
        if "정보 부족으로 작성 불가" in content:
             return {"generated_overview": "정보를 찾을 수 없습니다."}
        return {"generated_overview": content}
    except Exception as e:
        print(f"Generation failed: {e}")
        return {"generated_overview": ""}


In [19]:

# 그래프 구성
workflow = StateGraph(EtlState)
workflow.add_node("search", search_node)
workflow.add_node("summarize", summarize_node)

workflow.set_entry_point("search")
workflow.add_edge("search", "summarize")
workflow.add_edge("summarize", END)


etl_app = workflow.compile()

visualize_graph(etl_app)
# 메인 ETL 함수

그래프 시각화 실패 (추가 종속성 필요): Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 400.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`
ASCII로 그래프 표시:
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +--------+   
  | search |   
  +--------+   
      *        
      *        
      *        
+-----------+  
| summarize |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [21]:

def run_etl_pipeline(limit=10):
    conn = None
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor(dictionary=True)

        # 1. 대상 조회 (overview가 없거나 빈 문자열)
        select_query = """
            SELECT content_id, title, addr1 
            FROM attractions 
            WHERE overview IS NULL OR overview = ''
            LIMIT %s
        """
        cursor.execute(select_query, (limit,))
        rows = cursor.fetchall()
        
        print(f"Found {len(rows)} items to update.")

        updated_count = 0
        
        for row in rows:
            print(f"Processing: {row['title']} ({row['content_id']})")
            
            # 초기 상태
            initial_state = {
                "content_id": row['content_id'],
                "title": row['title'],
                "address": row['addr1'] or "",
                "search_results": "",
                "generated_overview": ""
            }
            
            # 워크플로우 실행
            final_state = etl_app.invoke(initial_state)
            overview = final_state.get("generated_overview", "")
            
            if overview and overview != "정보를 찾을 수 없습니다.":
                # DB 업데이트
                update_query = "UPDATE attractions SET overview = %s WHERE content_id = %s"
                cursor.execute(update_query, (overview, row['content_id']))
                conn.commit()
                updated_count += 1
                print(f"Updated: {row['title']}")
                print(f"Preview: {overview}")
            else:
                print(f"Skipped: {row['title']} (Low quality result)")
            
            # API 호출 제한 고려하여 약간의 지연
            time.sleep(1) # DuckDuckGo는 비율 제한이 있을 수 있으므로 2초로 넉넉하게

        print(f"ETL Complete. {updated_count}/{len(rows)} updated.")

    except mysql.connector.Error as err:
        print(f"Database Error: {err}")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if conn and conn.is_connected():
            cursor.close()
            conn.close()
run_etl_pipeline(limit=36)

Found 36 items to update.
Processing: 포베오커피 (2904102)
Searching for: 대한민국 전라남도 함평군 주포로 395 포베오커피 관광지 역사 특징 소개
Updated: 포베오커피
Preview: **포베오커피, 함평 바다를 품은 아늑한 커피 명소**  
포베오커피는 전라남도 함평군 주포로 395에 위치한 바다를 바라보며 커피를 즐길 수 있는 특별한 카페입니다.  

**본문**  
이 카페는 한적한 해안가 근처에 자리해 있어, 푸르른 바다와 어우러진 풍경이 가장 큰 매력입니다. 실내에는 넓고 편안한 좌석 공간이 마련되어 있어 가족, 연인, 친구와 함께 여유로운 시간을 보내기에 적합합니다. 특히 창가 자리를 선택하면 푸른 바다를 배경으로 커피를 음미할 수 있어 사진 촬영 명소로도 인기가 높습니다.  

포베오커피는 신선한 원두로 내린 핸드드립 커피와 함께 베이커리 메뉴를 선보입니다. 지역 특산물이나 신선한 재료를 활용한 디저트도 함께 즐길 수 있어, 커피와 달콤한 조화가 일품입니다. 인근에는 횟집과 전통 음식점들이 밀집해 있어, 커피 한 잔으로 여유를 채운 뒤 신선한 해산물 식사까지 이어지는 여행 코스로도 추천됩니다.  

**마무리**  
방문객들에게는 오전 개장 시간을 노려 한적한 분위기에서 바다를 감상하며 커피를 마시는 것을 추천합니다. 날씨가 좋은 날에는 야외 테라스가 열려 더욱 오픈된 공간에서 자연의 정취를 느낄 수 있습니다. 또한, 인근에 있는 주포 수산시장이나 역사 깊은 음식점들과 함께 방문하면 함평의 정취를 고스란히 체험할 수 있습니다. 포베오커피는 단순한 카페를 넘어 함평의 자연과 문화를 경험하는 작은 휴식처입니다.  

(※ 참고: 제공된 검색 결과에는 포베오커피에 대한 구체적 정보가 부족해, 함평 지역의 분위기와 인접 관광자원을 기반으로 일반적인 커피 명소의 특징을 반영해 작성하였습니다.)
Processing: 푸른산장 (2906065)
Searching for: 대한민국 전라남도 광양시 신재로 1